# 多源融合机器人定位及任务优化：四问全结果复现

本 Notebook 以 00_problem/attachments 中的四个原始附件和结果模板为唯一外部输入，
依次调用问题一至问题四的正式程序入口，重算主结果、灵敏度分析、正式图件及机器校验。
复现产物写入 04_experiments/reproduction_run，不覆盖 05_results，也不修改原始附件。

## 0. 计算约定

问题一把加到方式 2 时间戳上的校正量记为
$$t_{2,\mathrm{ref}}=t_2+\Delta t_c.$$

问题二、三采用观测模型中的时间偏差
$$t_{2,\mathrm{ref}}=t_2-\Delta t,\qquad
\mathbf z^{(2)}(t)=\mathbf r(t-\Delta t)+\mathbf b+\boldsymbol\varepsilon^{(2)}.$$

连续量按数值容差比较，任务数、偏差状态、轨迹行数和工作簿行数必须完全一致。

In [1]:
from pathlib import Path
import json
import shutil
import subprocess
import sys
import time

import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path.cwd().resolve()
if not (ROOT / "03_models").is_dir():
    raise RuntimeError("请从仓库根目录启动并执行本 Notebook。")
RUN_DIR = (ROOT / "04_experiments" / "reproduction_run").resolve()
if RUN_DIR.parent != (ROOT / "04_experiments").resolve():
    raise RuntimeError("复现输出目录越出预期范围。")
if RUN_DIR.exists():
    shutil.rmtree(RUN_DIR)
RESULTS, FIGURES = RUN_DIR / "results", RUN_DIR / "figures"
RESULTS.mkdir(parents=True)
FIGURES.mkdir(parents=True)
ATT = ROOT / "00_problem" / "attachments"
SHEET1, SHEET2 = "方式1(4Hz)", "方式2(5Hz)"

def run_step(label, *args):
    started = time.perf_counter()
    proc = subprocess.run(
        [sys.executable, *map(str, args)], cwd=ROOT, text=True,
        encoding="utf-8", errors="backslashreplace", capture_output=True
    )
    elapsed = time.perf_counter() - started
    if proc.returncode:
        print(proc.stdout[-4000:])
        print(proc.stderr[-4000:])
        raise RuntimeError(f"{label} 失败，退出码 {proc.returncode}")
    print(f"[PASS] {label}: {elapsed:.1f} s")
    return elapsed

print("Python:", sys.version.split()[0])
print("仓库:", ROOT)
print("复现输出:", RUN_DIR)

Python: 3.12.13
仓库: D:\MathModeling\_upload_AFMB_main_20260821
复现输出: D:\MathModeling\_upload_AFMB_main_20260821\04_experiments\reproduction_run


## 1. 问题一：无噪声时间配准与 10 Hz 轨迹

三次样条给出连续轨迹，速度特征的归一化互相关生成粗候选，最终最小化位置均方误差：
$$
J(\delta)=\frac{1}{|\Omega(\delta)|}\sum_{t_k\in\Omega(\delta)}
\|\mathbf r_1(t_k)-\mathbf r_2(t_k-\delta)\|_2^2,\qquad
\widehat{\Delta t_c}=\arg\min_{\delta\in\mathcal D}J(\delta).
$$
校正后合并同时刻观测，并仅在原始观测支持域内重建 10 Hz 轨迹。

In [2]:
q1r, q1f = RESULTS / "q1", FIGURES / "q1"
run_step("Q1 主链", ROOT / "03_models/q1/run_q1.py",
         "--input", ATT / "附件1.xlsx", "--output-dir", q1r,
         "--figure-dir", q1f, "--method", "cubic", "--compare-interpolators")
run_step("Q1 灵敏度", ROOT / "03_models/q1/run_sensitivity.py",
         "--input", ATT / "附件1.xlsx", "--output", q1r / "sensitivity.csv")
q1 = json.loads((q1r / "parameters.json").read_text(encoding="utf-8"))
pd.DataFrame([{"time_correction_s": q1["time_offset_s"],
               "aligned_rmse_m": q1["loss"]["rmse"],
               "output_rows": len(pd.read_csv(q1r / "trajectory_10hz.csv")),
               "validation_passed": q1["validation"]["passed"]}])

[PASS] Q1 主链: 22.3 s


[PASS] Q1 灵敏度: 8.9 s


,time_correction_s,aligned_rmse_m,output_rows,validation_passed
0,-198.4317,8.514815e-11,8495,True


## 2. 问题二：稳健时空标定与异步融合

固定公共网格避免候选偏差改变评价样本。给定 $\delta$，逐轴 Huber 估计相对空间偏差：
$$
\widehat{\mathbf b}(\delta)=\arg\min_{\mathbf b}\sum_k
\rho_{1.345}\left(
\frac{\mathbf z_2(t_k+\delta)-\mathbf z_1(t_k)-\mathbf b}{\mathbf s_r}\right),
\qquad \widehat{\Delta t}=\arg\min_\delta J_H(\delta).
$$
校正后的原始异步事件进入常加速度 Kalman 滤波与 RTS 平滑，10 Hz 网格只用于输出。

In [3]:
q2r, q2f = RESULTS / "q2", FIGURES / "q2"
run_step("Q2 主链", ROOT / "03_models/q2/run_q2.py", ATT / "附件2.xlsx",
         "--sheet1", SHEET1, "--sheet2", SHEET2,
         "--output", q2r / "trajectory_10hz.csv",
         "--summary", q2r / "parameters.json",
         "--innovations", q2r / "innovations.csv",
         "--tuning", q2r / "process_noise_tuning.csv")
run_step("Q2 灵敏度", ROOT / "03_models/q2/run_sensitivity.py",
         "--input", ATT / "附件2.xlsx", "--sheet1", SHEET1, "--sheet2", SHEET2,
         "--parameters", q2r / "parameters.json",
         "--trajectory", q2r / "trajectory_10hz.csv",
         "--output", q2r / "sensitivity.csv")
run_step("Q2 图件", ROOT / "03_models/q2/make_figures.py",
         "--input", ATT / "附件2.xlsx", "--sheet1", SHEET1, "--sheet2", SHEET2,
         "--results", q2r, "--figures", q2f)
run_step("Q2 机器校验", ROOT / "03_models/q2/validation.py",
         "--results", q2r, "--figures", q2f, "--output", q2r / "validation.json")
q2 = json.loads((q2r / "parameters.json").read_text(encoding="utf-8"))
pd.DataFrame([{k: q2[k] for k in
               ("time_offset_s", "bias_x_m", "bias_y_m", "mean_nis", "output_rows")}])

[PASS] Q2 主链: 34.8 s


[PASS] Q2 灵敏度: 24.4 s


[PASS] Q2 图件: 10.8 s


[PASS] Q2 机器校验: 1.2 s


,time_offset_s,bias_x_m,bias_y_m,mean_nis,output_rows
0,50.171732,3.466998,-1.8332,1.964694,8496


## 3. 问题三：偏差检验驱动的状态模型选择

时间配准后的残差为
$$\mathbf d_k=\widetilde{\mathbf z}_2(t_k+\widehat{\Delta t})
-\widetilde{\mathbf z}_1(t_k).$$
HAC 协方差修正序列相关，Wald 统计量为
$$W=\overline{\mathbf d}^{\mathsf T}
\widehat{\operatorname{Cov}}_{\rm HAC}(\overline{\mathbf d})^{-1}
\overline{\mathbf d}\sim\chi^2_2.$$
统计显著性与工程效应阈值共同决定是否启用偏差状态，移动块 Bootstrap 与趋势检验用于互证。

In [4]:
q3r, q3f = RESULTS / "q3", FIGURES / "q3"
run_step("Q3 主链", ROOT / "03_models/q3/run_q3.py", ATT / "附件3.xlsx",
         "--sheet1", SHEET1, "--sheet2", SHEET2,
         "--output", q3r / "trajectory_10hz.csv",
         "--summary", q3r / "parameters.json",
         "--innovations", q3r / "innovations.csv",
         "--tuning", q3r / "process_noise_tuning.csv")
run_step("Q3 灵敏度", ROOT / "03_models/q3/run_sensitivity.py",
         "--input", ATT / "附件3.xlsx", "--sheet1", SHEET1, "--sheet2", SHEET2,
         "--parameters", q3r / "parameters.json",
         "--trajectory", q3r / "trajectory_10hz.csv",
         "--output", q3r / "sensitivity.csv")
run_step("Q3 图件", ROOT / "03_models/q3/make_figures.py",
         "--input", ATT / "附件3.xlsx", "--sheet1", SHEET1, "--sheet2", SHEET2,
         "--results", q3r, "--figures", q3f)
run_step("Q3 机器校验", ROOT / "03_models/q3/validation.py",
         "--results", q3r, "--figures", q3f, "--output", q3r / "validation.json")
q3 = json.loads((q3r / "parameters.json").read_text(encoding="utf-8"))
pd.DataFrame([{"time_offset_s": q3["time_offset_s"],
               "wald_p_value": q3["wald"]["p_value"],
               "effect_index": q3["wald"]["effect_index"],
               "bias_state_enabled": q3["bias_state_enabled"],
               "output_rows": q3["output_rows"]}])

[PASS] Q3 主链: 16.6 s


[PASS] Q3 灵敏度: 12.1 s


[PASS] Q3 图件: 9.5 s


[PASS] Q3 机器校验: 1.2 s


,time_offset_s,wald_p_value,effect_index,bias_state_enabled,output_rows
0,-367.877619,0.351076,0.068081,False,3691


## 4. 问题四：连续可行窗口与无次数上限调度

候选 $c$ 占用完整准备区间 $I_c$，其归一化最小裕度为
$$m_c=\min_{t\in I_c}\min_r
\frac{g_r(t)-g_r^{\min}}{g_r^{\max}-g_r^{\min}}.$$
令 $x_c\in\{0,1\}$。在任务资源互斥、射击目标唯一和同一拍照目标角差不少于
$60^\circ$ 的约束下，不设置固定任务次数上限，依次最大化任务数、最小裕度和总裕度。
本节读取刚刚重算的问题三轨迹，以验证跨问数据链。

In [5]:
q4r, q4f = RESULTS / "q4", FIGURES / "q4"
run_step("Q4 主链", ROOT / "03_models/q4/run_q4.py",
         "--trajectory", q3r / "trajectory_10hz.csv",
         "--targets", ATT / "附件4.xlsx", "--template", ATT / "result.xlsx",
         "--results", q4r)
run_step("Q4 图件", ROOT / "03_models/q4/make_figures.py",
         "--trajectory", q3r / "trajectory_10hz.csv",
         "--targets", ATT / "附件4.xlsx", "--results", q4r, "--figures", q4f)
run_step("Q4 机器校验", ROOT / "03_models/q4/validation.py",
         "--trajectory", q3r / "trajectory_10hz.csv",
         "--targets", ATT / "附件4.xlsx", "--template", ATT / "result.xlsx",
         "--results", q4r, "--figures", q4f,
         "--output", q4r / "validation.json")
q4 = json.loads((q4r / "parameters.json").read_text(encoding="utf-8"))
pd.DataFrame([{k: q4[k] for k in
               ("maximum_task_count", "shooting_count", "photography_count",
                "expected_shooting_hits", "greedy_task_count")}])

[PASS] Q4 主链: 15.6 s


[PASS] Q4 图件: 13.1 s


[PASS] Q4 机器校验: 7.2 s


,maximum_task_count,shooting_count,photography_count,expected_shooting_hits,greedy_task_count
0,40,14,26,11.9,38


## 5. 复现门禁

不比较绝对路径、运行时版本或图件二进制哈希，因为它们允许随计算机环境改变。
仅比较论文结论依赖的连续参数和离散决策。

In [6]:
reference = {
    q: json.loads((ROOT / "05_results" / q / "parameters.json").read_text(encoding="utf-8"))
    for q in ("q1", "q2", "q3", "q4")
}
checks = []
def close_check(question, metric, actual, expected, atol):
    checks.append({"question": question, "metric": metric, "actual": actual,
                   "reference": expected, "tolerance": atol,
                   "passed": bool(np.isclose(actual, expected, rtol=0, atol=atol))})
def exact_check(question, metric, actual, expected):
    checks.append({"question": question, "metric": metric, "actual": actual,
                   "reference": expected, "tolerance": "exact",
                   "passed": bool(actual == expected)})

close_check("Q1", "time_offset_s", q1["time_offset_s"], reference["q1"]["time_offset_s"], 5e-6)
close_check("Q1", "aligned_rmse_m", q1["loss"]["rmse"], reference["q1"]["loss"]["rmse"], 1e-8)
exact_check("Q1", "output_rows", len(pd.read_csv(q1r / "trajectory_10hz.csv")),
            len(pd.read_csv(ROOT / "05_results/q1/trajectory_10hz.csv")))
for metric in ("time_offset_s", "bias_x_m", "bias_y_m"):
    close_check("Q2", metric, q2[metric], reference["q2"][metric], 5e-5)
exact_check("Q2", "output_rows", q2["output_rows"], reference["q2"]["output_rows"])
close_check("Q3", "time_offset_s", q3["time_offset_s"], reference["q3"]["time_offset_s"], 5e-4)
close_check("Q3", "wald_p_value", q3["wald"]["p_value"],
            reference["q3"]["wald"]["p_value"], 5e-4)
exact_check("Q3", "bias_state_enabled", q3["bias_state_enabled"],
            reference["q3"]["bias_state_enabled"])
exact_check("Q3", "output_rows", q3["output_rows"], reference["q3"]["output_rows"])
for metric in ("maximum_task_count", "shooting_count", "photography_count", "greedy_task_count"):
    exact_check("Q4", metric, q4[metric], reference["q4"][metric])
close_check("Q4", "expected_shooting_hits", q4["expected_shooting_hits"],
            reference["q4"]["expected_shooting_hits"], 1e-12)
exact_check("Q4", "result_filled_rows", q4["result_workbook"]["filled_rows"],
            reference["q4"]["result_workbook"]["filled_rows"])
for question, directory in (("Q2", q2r), ("Q3", q3r), ("Q4", q4r)):
    report = json.loads((directory / "validation.json").read_text(encoding="utf-8"))
    exact_check(question, "machine_validation", report["ok"], True)

check_table = pd.DataFrame(checks)
display(check_table)
all_passed = bool(check_table["passed"].all())
summary = {"all_passed": all_passed, "checks": checks,
           "output_directory": str(RUN_DIR.relative_to(ROOT))}
(RUN_DIR / "reproduction_summary.json").write_text(
    json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")
if not all_passed:
    raise AssertionError("至少一项复现门禁未通过。")
print(f"全部 {len(checks)} 项复现门禁通过。")

,question,metric,actual,reference,tolerance,passed
0,Q1,time_offset_s,-198.4317,-198.4317,0.000005,True
1,Q1,aligned_rmse_m,0.0,0.0,0.0,True
2,Q1,output_rows,8495,8495,exact,True
3,Q2,time_offset_s,50.171732,50.171732,0.00005,True
4,Q2,bias_x_m,3.466998,3.466998,0.00005,True
5,Q2,bias_y_m,-1.8332,-1.8332,0.00005,True
6,Q2,output_rows,8496,8496,exact,True
7,Q3,time_offset_s,-367.877619,-367.877619,0.0005,True
8,Q3,wald_p_value,0.351076,0.351076,0.0005,True
9,Q3,bias_state_enabled,False,False,exact,True


全部 20 项复现门禁通过。


## 6. 结论

上一单元通过即表明四问已从原始附件顺序重算，问题二、三的统计与滤波产物通过机器校验，
问题四使用了本次重算的问题三轨迹，且没有固定九项任务上限。